# STEP2. 리뷰 감성 분석 데모 (Gradio)

STEP1에서 저장한 감성 모델 두 개를 불러와, **문장을 넣으면 바로 결과가 나오는 화면**을 만듭니다.

| 모델 | 파일 | 노트북 |
|---|---|---|
| 머신러닝 (TF-IDF + 로지스틱 회귀) | `part7_1_sentiment_ml.pkl` | 07_1 |
| 딥러닝 (Embedding) | `part7_2_sentiment_model.keras` + `part7_2_tokenizer.pkl` | 07_2 |

```
문장 입력 → 모델 2개가 각각 판정 → 화면에 긍정/부정 확률
```

> 딥러닝 모델은 TensorFlow가 필요합니다. Colab에는 이미 설치되어 있어서 여기서는 바로 쓸 수 있습니다.
> STEP3 웹앱(내 컴퓨터 · 배포)에서는 설치가 무거워 머신러닝 모델만 사용합니다.

In [1]:
# 셀 1. Drive 연결 + 경로
from google.colab import drive
drive.mount('/content/drive')

base_path  = "/content/drive/MyDrive/Colab Notebooks/Data_Analysis/Olist"
model_path = f"{base_path}/model"     # STEP1에서 모델을 저장한 곳

Mounted at /content/drive


In [2]:
# 셀 2. Gradio 설치 (Colab에 없으면 설치)
!pip install -q gradio

In [3]:
# 셀 3. 모델 불러오기
import pickle
import joblib
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

ml = joblib.load(f"{model_path}/part7_1_sentiment_ml.pkl")              # {"tfidf", "model"}
dl = load_model(f"{model_path}/part7_2_sentiment_model.keras")
with open(f"{model_path}/part7_2_tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)
MAX_LEN = 50                                                             # 07_2 학습 때와 같은 길이
print("모델 불러오기 완료!")

모델 불러오기 완료!


In [4]:
# 셀 4. 예측 함수 (각 노트북 마지막 '저장한 파일로 예측해 보기' 셀과 같은 순서)
def predict_ml(text):
    return float(ml["model"].predict_proba(ml["tfidf"].transform([text]))[0, 1])   # 긍정 확률

def predict_dl(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    return float(dl.predict(pad, verbose=0)[0][0])                                  # 긍정 확률

text = "produto veio quebrado, péssimo"          # 상품이 깨져서 왔다, 최악
print(f"머신러닝 긍정 확률 : {predict_ml(text):.3f}")
print(f"딥러닝   긍정 확률 : {predict_dl(text):.3f}")

머신러닝 긍정 확률 : 0.006
딥러닝   긍정 확률 : 0.021


## Gradio 화면 만들기

`gr.Interface`에 **함수 · 입력 칸 · 출력 칸** 세 가지만 알려 주면 화면이 만들어집니다.

| 넣는 것 | 이 노트북에서 |
|---|---|
| `fn` (함수) | `analyze` : 문장 → 두 모델의 확률 |
| `inputs` (입력 칸) | 글 상자 1개 |
| `outputs` (출력 칸) | 확률 막대 2개 (`gr.Label`) |

In [5]:
# 셀 5. 화면 만들고 실행
import gradio as gr

def analyze(text):
    p_ml, p_dl = predict_ml(text), predict_dl(text)
    return ({"긍정": p_ml, "부정": 1 - p_ml},        # gr.Label은 {이름: 확률}을 막대로 보여 줍니다
            {"긍정": p_dl, "부정": 1 - p_dl})

demo = gr.Interface(
    fn=analyze,
    inputs=gr.Textbox(label="리뷰 (포르투갈어)", lines=3),
    outputs=[gr.Label(label="머신러닝 (07_1)"), gr.Label(label="딥러닝 (07_2)")],
    examples=["produto muito bom, chegou antes do prazo",   # 상품 아주 좋음, 일찍 도착
              "ainda não recebi o produto",                 # 아직 못 받았다
              "produto veio quebrado, péssimo"],            # 깨져서 왔다, 최악
    title="리뷰 감성 분석 데모",
)
demo.launch()          # 셀 아래에 화면이 나타납니다. (public URL이 뜨면 다른 기기에서도 접속 가능)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://11593ed3caf40cba70.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 해 볼 것
1. 예시 문장 3개를 눌러 두 모델의 결과를 비교해 보세요. 대부분 비슷하게 나옵니다.
2. 한국어나 영어 문장을 넣어 보세요. 포르투갈어로만 학습했기 때문에 결과가 엉뚱합니다.
3. 이 화면은 Colab 런타임이 켜져 있는 동안만 열립니다. 런타임을 끊으면 사라집니다.

> 👉 **STEP3**에서는 Colab을 끄면 사라지는 이 화면 대신, 계속 열 수 있는 **Streamlit 웹앱**을 만듭니다.